# TP3: Chatbot de viajes cortos RumboSur

RumboSur es una empresa ficticia que ofrece viajes cortos entre Río Cuarto, Córdoba, Villa María y Villa Carlos Paz. El chatbot responde consultas sobre horarios, días, precios, duración, disponibilidad y trayectos.

Se implementan dos chatbots basados en recuperación de información: uno con TF-IDF y otro con embeddings.

## 1. Preparación del entorno

In [2]:
!pip install -q spacy scikit-learn
!python -m spacy download es_core_news_sm -q

import unicodedata
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nlp = spacy.load('es_core_news_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Dataset de preguntas y respuestas



In [3]:
datos = [
 ('¿Qué viajes ofrecen?', 'Ofrecemos viajes entre Río Cuarto, Córdoba, Villa María y Villa Carlos Paz.'),
 ('¿Qué destinos tienen disponibles?', 'Los destinos disponibles son Córdoba, Villa María y Villa Carlos Paz.'),
 ('¿Puedo viajar desde Río Cuarto a Córdoba?', 'Sí, existe el trayecto Río Cuarto-Córdoba.'),
 ('¿Hay viajes de Córdoba a Río Cuarto?', 'Sí, también ofrecemos el trayecto Córdoba-Río Cuarto.'),
 ('¿Cuándo sale el colectivo a Córdoba?', 'De Río Cuarto a Córdoba sale de lunes a viernes a las 06:30.'),
 ('¿A qué hora parte el viaje Río Cuarto-Córdoba?', 'Sale a las 06:30 de lunes a viernes.'),
 ('¿Cuándo regresa el servicio desde Córdoba?', 'De Córdoba a Río Cuarto sale de lunes a viernes a las 18:00.'),
 ('¿Qué días viajan a Villa María?', 'A Villa María viajamos los martes, jueves y sábados.'),
 ('¿Hay frecuencias para Villa María?', 'Sí, hay servicios los martes, jueves y sábados.'),
 ('¿A qué hora sale el viaje a Villa María?', 'El servicio Río Cuarto-Villa María sale a las 08:00.'),
 ('¿A qué hora vuelve desde Villa María?', 'El regreso Villa María-Río Cuarto sale a las 17:00.'),
 ('¿Qué días hay viajes a Carlos Paz?', 'A Villa Carlos Paz viajamos los viernes y domingos.'),
 ('¿Hay un colectivo a Carlos Paz el domingo?', 'Sí, hay servicio a Villa Carlos Paz los domingos a las 07:00.'),
 ('¿A qué hora sale para Villa Carlos Paz?', 'El viaje Río Cuarto-Villa Carlos Paz sale a las 07:00.'),
 ('¿A qué hora regresa desde Carlos Paz?', 'El regreso Villa Carlos Paz-Río Cuarto sale a las 18:30.'),
 ('¿Cuánto cuesta ir a Córdoba?', 'El pasaje entre Río Cuarto y Córdoba cuesta 8500 pesos.'),
 ('¿Cuál es el precio de Villa María?', 'El pasaje a Villa María cuesta 6500 pesos.'),
 ('¿Cuánto sale el pasaje a Carlos Paz?', 'El pasaje a Villa Carlos Paz cuesta 9000 pesos.'),
 ('¿Cuánto demora el viaje a Córdoba?', 'El viaje entre Río Cuarto y Córdoba dura aproximadamente 3 horas.'),
 ('¿Cuánto tarda el viaje a Villa María?', 'El viaje a Villa María dura aproximadamente 2 horas.'),
 ('¿Cuánto dura el viaje a Carlos Paz?', 'El viaje a Villa Carlos Paz dura aproximadamente 3 horas y 30 minutos.'),
 ('¿Quedan lugares para Córdoba?', 'Sí, quedan 12 lugares en el servicio Río Cuarto-Córdoba.'),
 ('¿Hay asientos disponibles para Villa María?', 'Sí, quedan 15 lugares para Villa María.'),
 ('¿Hay disponibilidad para Carlos Paz?', 'Sí, quedan 10 lugares para Villa Carlos Paz.')
]
preguntas = [p for p, r in datos]
respuestas = [r for p, r in datos]
dataset = pd.DataFrame({'pregunta': preguntas, 'respuesta': respuestas})
dataset.head()

,pregunta,respuesta
0,¿Qué viajes ofrecen?,"Ofrecemos viajes entre Río Cuarto, Córdoba, Vi..."
1,¿Qué destinos tienen disponibles?,"Los destinos disponibles son Córdoba, Villa Ma..."
2,¿Puedo viajar desde Río Cuarto a Córdoba?,"Sí, existe el trayecto Río Cuarto-Córdoba."
3,¿Hay viajes de Córdoba a Río Cuarto?,"Sí, también ofrecemos el trayecto Córdoba-Río ..."
4,¿Cuándo sale el colectivo a Córdoba?,De Río Cuarto a Córdoba sale de lunes a vierne...


## 3. Limpieza básica del texto



In [4]:
def normalizar(texto):
    texto = texto.lower()
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

preguntas_limpias = [normalizar(p) for p in preguntas]

## 4. Chatbot con TF-IDF



In [5]:
vectorizador = TfidfVectorizer(strip_accents='unicode')
matriz_tfidf = vectorizador.fit_transform(preguntas_limpias)

def responder_tfidf(consulta, umbral=0.10, devolver_indice=False):
    consulta_vector = vectorizador.transform([normalizar(consulta)])
    similitudes = cosine_similarity(consulta_vector, matriz_tfidf)[0]
    indice = int(np.argmax(similitudes))
    if similitudes[indice] < umbral:
        respuesta = 'No encontré una respuesta para esa consulta. Intenta preguntar por un trayecto, horario, precio o disponibilidad.'
    else:
        respuesta = respuestas[indice]
    if devolver_indice:
        return respuesta, indice, float(similitudes[indice])
    return respuesta

responder_tfidf('¿Cuánto cuesta viajar a Córdoba?')

'El pasaje entre Río Cuarto y Córdoba cuesta 8500 pesos.'

## 5. Chatbot con embeddings



In [6]:
def vector_embedding(texto):
    return nlp(texto).vector

matriz_embeddings = np.vstack([vector_embedding(p) for p in preguntas_limpias])

def responder_embeddings(consulta, umbral=0.10, devolver_indice=False):
    consulta_vector = vector_embedding(normalizar(consulta)).reshape(1, -1)
    similitudes = cosine_similarity(consulta_vector, matriz_embeddings)[0]
    indice = int(np.argmax(similitudes))
    if similitudes[indice] < umbral:
        respuesta = 'No encontré una respuesta para esa consulta. Intenta preguntar por un trayecto, horario, precio o disponibilidad.'
    else:
        respuesta = respuestas[indice]
    if devolver_indice:
        return respuesta, indice, float(similitudes[indice])
    return respuesta

responder_embeddings('¿Hay asientos para ir a Carlos Paz?')

'Sí, quedan 10 lugares para Villa Carlos Paz.'

## 6. Pruebas

In [7]:
pruebas = [
    ('¿Cuál es el valor del boleto a Córdoba?', 15),
    ('¿A qué hora parte el micro para Villa María?', 9),
    ('¿Hay asientos libres para viajar a Carlos Paz?', 23),
    ('¿Cuánto demora llegar a Villa María?', 19),
    ('¿Qué días puedo ir a Córdoba?', 4)
]

resultados = []
for consulta, indice_esperado in pruebas:
    r_tfidf, i_tfidf, s_tfidf = responder_tfidf(consulta, devolver_indice=True)
    r_emb, i_emb, s_emb = responder_embeddings(consulta, devolver_indice=True)
    resultados.append({
        'consulta': consulta,
        'respuesta_tfidf': r_tfidf,
        'respuesta_embeddings': r_emb,
        'tfidf_acierta': i_tfidf == indice_esperado,
        'embeddings_acierta': i_emb == indice_esperado,
        'similitud_tfidf': round(s_tfidf, 3),
        'similitud_embeddings': round(s_emb, 3)
    })

evaluacion = pd.DataFrame(resultados)
evaluacion

,consulta,respuesta_tfidf,respuesta_embeddings,tfidf_acierta,embeddings_acierta,similitud_tfidf,similitud_embeddings
0,¿Cuál es el valor del boleto a Córdoba?,El pasaje a Villa María cuesta 6500 pesos.,El pasaje a Villa María cuesta 6500 pesos.,False,False,0.646,0.854
1,¿A qué hora parte el micro para Villa María?,El servicio Río Cuarto-Villa María sale a las ...,El servicio Río Cuarto-Villa María sale a las ...,True,True,0.583,0.905
2,¿Hay asientos libres para viajar a Carlos Paz?,"Sí, quedan 15 lugares para Villa María.","Sí, quedan 10 lugares para Villa Carlos Paz.",False,True,0.541,0.795
3,¿Cuánto demora llegar a Villa María?,El viaje entre Río Cuarto y Córdoba dura aprox...,El viaje a Villa María dura aproximadamente 2 ...,False,True,0.627,0.802
4,¿Qué días puedo ir a Córdoba?,El pasaje entre Río Cuarto y Córdoba cuesta 85...,El pasaje entre Río Cuarto y Córdoba cuesta 85...,False,False,0.430,0.713


In [8]:
print('Precisión TF-IDF:', evaluacion['tfidf_acierta'].mean())
print('Precisión embeddings:', evaluacion['embeddings_acierta'].mean())

Precisión TF-IDF: 0.2
Precisión embeddings: 0.6


## 7. Conclusiones

TF-IDF funciona bien cuando las palabras de la consulta se parecen a las preguntas almacenadas. Es sencillo, rápido e interpretable, pero puede fallar ante sinónimos o formulaciones muy diferentes.

Los embeddings buscan una relación semántica entre los textos y pueden reconocer consultas parecidas aunque no compartan exactamente las mismas palabras. Su resultado depende del modelo utilizado y, en este caso, del tamaño del modelo de spaCy.

La comparación debe realizarse observando tanto las respuestas como las medidas de similitud y la precisión obtenida en las pruebas. Para una aplicación real habría que actualizar horarios, precios y disponibilidad en tiempo real.